# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 7 周：本量利分析与存货管理

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 掌握贡献毛益、C/S ratio 与盈亏平衡点的计算
2. 理解安全边际、目标利润与敏感性分析
3. 掌握 EOQ 经济订货量模型的推导与应用
4. 用 Python 实现 CVP 计算、EOQ 求解与可视化

In [ ]:
import numpy as np                                # 数值计算
import matplotlib.pyplot as plt                   # 绘图

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False        # 负号显示

## 1. CVP 分析：咖啡馆盈亏平衡

利润方程 $\pi = (P - v)Q - F = CM \cdot Q - F$，令 $\pi = 0$ 得盈亏平衡点：

$$Q_{BE} = \frac{F}{CM}, \qquad S_{BE} = \frac{F}{\text{C/S ratio}}$$

In [ ]:
P, v, F = 35, 15, 8000                            # 单价35元/杯、变动成本15元/杯、月固定成本8000元

CM = P - v                                        # 单位贡献毛益 = 35-15 = 20 元/杯
CS_ratio = CM / P                                 # 贡献毛益率 C/S ratio = 20/35 ≈ 57.14%
Q_BE = F / CM                                     # 盈亏平衡销量 = 8000/20 = 400 杯
S_BE = F / CS_ratio                               # 盈亏平衡销售额 = 8000/0.5714 = 14000 元

print(f'单位贡献毛益 CM = {CM} 元/杯')               # 打印 CM
print(f'贡献毛益率 C/S = {CS_ratio:.2%}')            # 打印 C/S ratio
print(f'盈亏平衡销量 = {Q_BE:.0f} 杯')               # 打印保本销量
print(f'盈亏平衡销售额 = {S_BE:.0f} 元')              # 打印保本销售额

In [ ]:
# ===== CVP 图：收入线与成本线 =====
Q = np.linspace(0, 800, 200)                      # 销量范围 0~800 杯
TR = P * Q                                        # 总收入线 = 单价×销量
TC = F + v * Q                                    # 总成本线 = 固定 + 单位变动×销量

plt.figure(figsize=(10, 6))                        # 画布
plt.plot(Q, TR, label='总收入 TR', color='#C49A6C', linewidth=2.5)   # 收入线
plt.plot(Q, TC, label='总成本 TC', color='#8B6F4E', linewidth=2.5)   # 成本线
plt.axvline(Q_BE, color='#E74C3C', linestyle='--', alpha=0.8, label=f'盈亏平衡 Q={Q_BE:.0f}')   # 保本量垂直线
plt.fill_between(Q, TC, TR, where=(TR >= TC), alpha=0.15, color='green', label='盈利区')   # 交点右侧填绿
plt.fill_between(Q, TC, TR, where=(TR < TC), alpha=0.15, color='red', label='亏损区')   # 交点左侧填红
plt.xlabel('销量 Q（杯）')                          # x 轴
plt.ylabel('金额（元）')                            # y 轴
plt.title('CVP 本量利图：咖啡馆案例')                # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.show()                                        # 显示

## 2. 安全边际与目标利润

$$MS\% = \frac{Q_{actual} - Q_{BE}}{Q_{actual}} \times 100\%, \qquad Q_{target} = \frac{F + \text{目标利润}}{CM}$$

In [ ]:
Q_actual = 600                                    # 实际月销量 600 杯
MS_units = Q_actual - Q_BE                        # 安全边际（数量）= 600-400 = 200 杯
MS_pct = MS_units / Q_actual * 100                # 安全边际率 = 200/600 ≈ 33.33%

target_profit = 4000                              # 目标月利润 4000 元
Q_target = (F + target_profit) / CM               # 目标销量 = (8000+4000)/20 = 600 杯

print(f'安全边际 = {MS_units:.0f} 杯（{MS_units*P:.0f} 元）')   # 数量与金额形式
print(f'安全边际率 = {MS_pct:.2f}%（销量最多可下滑三成仍不亏损）')   # 比率形式解读
print(f'实现 {target_profit} 元目标利润需销量 = {Q_target:.0f} 杯')   # 目标销量（恰为实际销量）

In [ ]:
# ===== 敏感性分析：价格/成本变动对盈亏平衡的影响 =====
price_changes = np.linspace(-20, 20, 41)          # 单价变动 ±20%（41 个点）
Q_BE_prices = [F / (P*(1+pc/100) - v) for pc in price_changes]   # 每个价格下的保本量

vc_changes = np.linspace(-20, 20, 41)             # 变动成本变动 ±20%
Q_BE_vcs = [F / (P - v*(1+vc/100)) for vc in vc_changes]   # 每个成本下的保本量

fig, axes = plt.subplots(1, 2, figsize=(14, 5))   # 1 行 2 列
axes[0].plot(price_changes, Q_BE_prices, color='#C49A6C', linewidth=2.5)   # 价格敏感性曲线
axes[0].axhline(Q_BE, color='#E74C3C', linestyle='--', label=f'基准 {Q_BE:.0f} 杯')   # 基准线
axes[0].set_xlabel('单价变动（%）')                 # x 轴
axes[0].set_ylabel('盈亏平衡销量（杯）')            # y 轴
axes[0].set_title('单价变动的影响')                 # 标题
axes[0].legend(); axes[0].grid(True, alpha=0.3)   # 图例与网格

axes[1].plot(vc_changes, Q_BE_vcs, color='#8B6F4E', linewidth=2.5)   # 成本敏感性曲线
axes[1].axhline(Q_BE, color='#E74C3C', linestyle='--', label=f'基准 {Q_BE:.0f} 杯')   # 基准线
axes[1].set_xlabel('变动成本变动（%）')             # x 轴
axes[1].set_ylabel('盈亏平衡销量（杯）')            # y 轴
axes[1].set_title('变动成本变动的影响')             # 标题
axes[1].legend(); axes[1].grid(True, alpha=0.3)   # 图例与网格
plt.tight_layout()                                # 布局
plt.show()                                        # 显示
print('规律：单价↑ → 保本点↓（更安全）；成本↑ → 保本点↑（更危险）')   # 总结规律

## 3. 多产品 CVP：加权平均法

$$S_{BE}^{total} = \frac{F}{w_A \cdot CS_A + w_B \cdot CS_B}$$

In [ ]:
# 文具店案例：精装本与简装本
F_multi = 10000                                   # 月固定成本 1 万元
w_hard, CS_hard = 0.60, (50-30)/50                # 精装本：销售额占比 60%，C/S = 40%
w_soft, CS_soft = 0.40, (20-12)/20                # 简装本：占比 40%，C/S = 40%

weighted_CS = w_hard * CS_hard + w_soft * CS_soft # 加权平均 C/S ratio = 0.4
S_BE_total = F_multi / weighted_CS                # 综合保本销售额 = 25000 元
S_BE_hard = S_BE_total * w_hard                   # 精装本保本额 = 15000
S_BE_soft = S_BE_total * w_soft                   # 简装本保本额 = 10000

check = S_BE_hard*CS_hard + S_BE_soft*CS_soft     # 验算：两产品贡献之和
print(f'加权 C/S = {weighted_CS:.0%}')              # 打印加权贡献毛益率
print(f'综合保本销售额 = {S_BE_total:.0f} 元')       # 打印综合保本额
print(f'精装本 {S_BE_hard:.0f} 元 / 简装本 {S_BE_soft:.0f} 元')   # 分产品保本额
print(f'验证：贡献合计 {check:.0f} = 固定成本 {F_multi} ✓')   # 验算通过

## 4. EOQ 经济订货量

总成本 $TC = \frac{D}{Q}S + \frac{Q}{2}H$，求导令其为零：

$$EOQ = \sqrt{\frac{2DS}{H}}, \qquad TC_{min} = \sqrt{2DSH}$$

**重要性质**：最优时年订货成本 = 年持有成本（可用于验算）。

In [ ]:
D, S_cost, H = 2400, 100, 6                       # 年需求2400本、每次订货100元、单位年持有成本6元

EOQ = np.sqrt(2 * D * S_cost / H)                 # EOQ = √(2×2400×100/6) = √80000 ≈ 283 本
order_times = D / EOQ                             # 年订货次数 ≈ 8.48 次
TC_min = np.sqrt(2 * D * S_cost * H)              # 最小总成本 ≈ 1697 元/年

order_cost = (D / EOQ) * S_cost                   # 年订货成本
holding_cost = (EOQ / 2) * H                      # 年持有成本

print(f'EOQ = {EOQ:.1f} 本')                       # 打印经济订货量
print(f'年订货次数 ≈ {order_times:.2f} 次')         # 打印订货频率
print(f'最小总成本 = {TC_min:.1f} 元/年')           # 打印最小总成本
print(f'验证：订货成本 {order_cost:.0f} ≈ 持有成本 {holding_cost:.0f} ✓')   # 两者近似相等

In [ ]:
# ===== EOQ 总成本曲线 =====
Q_range = np.linspace(50, 800, 300)               # 订货量扫描范围
TC_total = (D / Q_range) * S_cost + (Q_range / 2) * H   # 每个订货量的总成本
TC_order = (D / Q_range) * S_cost                 # 订货成本分量（随Q递减）
TC_hold = (Q_range / 2) * H                       # 持有成本分量（随Q递增）

plt.figure(figsize=(10, 6))                        # 画布
plt.plot(Q_range, TC_order, '--', color='#8B6F4E', alpha=0.7, label='年订货成本（递减）')   # 订货成本线
plt.plot(Q_range, TC_hold, '--', color='#C49A6C', alpha=0.7, label='年持有成本（递增）')   # 持有成本线
plt.plot(Q_range, TC_total, color='#E74C3C', linewidth=2.5, label='总成本 TC')   # 总成本线
plt.axvline(EOQ, color='gray', linestyle=':', label=f'EOQ = {EOQ:.0f} 本')   # EOQ 垂线
plt.scatter([EOQ], [TC_min], color='#E74C3C', s=100, zorder=5)   # 最优点标记
plt.xlabel('订货量 Q（本）')                        # x 轴
plt.ylabel('年成本（元）')                          # y 轴
plt.title('EOQ 模型：两成本此消彼长，交点即最优')    # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.show()                                        # 显示

In [ ]:
# ===== 存货水平「红绿灯」：再订货点 / 最低 / 最高库存 =====
max_daily, avg_daily, min_daily = 12, 8, 6        # 最大/平均/最小日用量（本/天）
max_lead, avg_lead, min_lead = 8, 5, 3            # 最长/平均/最短提前期（天）

RL = max_daily * max_lead                         # 再订货点 = 最大日用量×最长提前期（保守防缺货）
min_level = RL - avg_daily * avg_lead             # 最低库存 = RL - 正常提前期消耗
max_level = RL + EOQ - min_daily * min_lead       # 最高库存 = RL + EOQ - 最短提前期消耗

print(f'再订货点 RL = {RL:.0f} 本（库存降到此线立即下单）')   # 打印 RL
print(f'最低库存 = {min_level:.0f} 本')              # 打印最低库存
print(f'最高库存 = {max_level:.0f} 本')              # 打印最高库存
print(f'管理含义：库存正常波动区间 [{min_level:.0f}, {max_level:.0f}] 本')   # 管理解读

## 5. 本周小结

| 指标 | 公式 | 要点 |
|---|---|---|
| 盈亏平衡销量 | $Q_{BE} = F/CM$ | 利润为零的销量 |
| 盈亏平衡销售额 | $S_{BE} = F/\text{C/S}$ | 金额形式 |
| 安全边际率 | $(Q_a - Q_{BE})/Q_a$ | 销量可下滑的安全空间 |
| 目标销量 | $(F+TP)/CM$ | 保本 + 目标利润 |
| EOQ | $\sqrt{2DS/H}$ | 订货成本 = 持有成本时最优 |